In [ ]:
# SETUP ENVIRONMENT
# Check GPU
!nvidia-smi

# Install ultralytics
!pip install ultralytics -q

# Connect Google Drive into Colab VM
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# UPLOAD DATASET
import os
import yaml

DATASET_PATH = '/content/drive/MyDrive/GroupProject_Seg'  # dataset path

print(f"Dataset path: {DATASET_PATH}")

# Check dataset's structure for train and val of img and label
# Why train/val: In Ultralytics YOLO dataset YAML, test: is explicitly optional, while train: and val: are the standard splits used in the config
print("\nCheck dataset's structure:")
for split in ['train', 'val']:
    img_dir = os.path.join(DATASET_PATH, 'images', split)
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)

    if os.path.exists(img_dir):
        n_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f" {split}/images: {n_imgs} images")
    else:
        print(f" Not found: {img_dir}")

    if os.path.exists(lbl_dir):
        n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
        print(f" {split}/labels: {n_lbls} file")
    else:
        print(f" Not found: {lbl_dir}")

In [ ]:
# MAKE FILE DATA.YAML
data_yaml_content = {
    'path': DATASET_PATH,              # dataset path
    'train': 'images/train',
    'val': 'images/val',

    'nc': 4,                           # number of classes
    'names': ['Stairs', 'crosswalk','sidewalk', 'tree-lined' ]      # CLASS NAME
}

# Save file yaml
yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"Done: {yaml_path}")
print("\n Content data.yaml:")
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
# LOAD MODEL YOLOV8X
from ultralytics import YOLO

model = YOLO('yolov8x-seg.pt')
print("Loaded YOLOv8x-seg (segmentation pretrained model)")

# Save RUN in FOLDER DATASET
RUNS_DIR  = os.path.join(DATASET_PATH, "runs")
EXP_NAME  = "yolov8x_seg_finetune"

WEIGHTS_DIR = os.path.join(RUNS_DIR, EXP_NAME, "weights")
LAST_CKPT   = os.path.join(WEIGHTS_DIR, "last.pt")
BEST_CKPT   = os.path.join(WEIGHTS_DIR, "best.pt")

# Save type:
# False = only last.pt/best.pt, last overwrite each epoch)
# True  = save epoch*.pt each epoch
SAVE_EVERY_EPOCH = False

# AUTO RESUME with CHECKPOINT
if os.path.exists(LAST_CKPT):
    print(f" Found checkpoint -> RESUME from: {LAST_CKPT}")
    model = YOLO(LAST_CKPT)
    results = model.train(resume=True)
else:
    print(" No checkpoint -> START from yolov8x-seg.pt")
    model = YOLO("yolov8x-seg.pt")

    results = model.train(
        data=yaml_path,
        epochs=100,     # Each epoch represents a full pass over the entire dataset
        imgsz=640,      # if rect=false => size img: 640 x 640
        batch=32,
        device=0,       # Train on GPU 0

        project=RUNS_DIR,
        name=EXP_NAME,
        exist_ok=True,      # allow overwritting on the same project/name folder

        cache=True,     # improve speed, but use more memory
        workers=8,      # worker thread

        # data augmentation
        hsv_h=0.015,    # hue
        hsv_s=0.7,      # saturation
        hsv_v=0.4,      # brightness jitter
        
        degrees=0.0,    # rotate image random
        translate=0.1,  # aiding in learning to detect partially visible objects
        scale=0.5,
        shear=0.0,      # mimic object view from different angles
        perspective=0.0,

        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,         # combine 4 training img into 1, simulating different composition 
        mixup=0.0,
        copy_paste=0.0,     # disable copy image

        patience=50,        # early stop if performance not improve
        save=True,
        save_period=(1 if SAVE_EVERY_EPOCH else 0),

        optimizer='auto',   # ultralytic auto pick optimizer
        lr0=0.01,           # init learning rate
        lrf=0.01,           # final LR = lr0 * lrf = 0.0001
        momentum=0.937,         # accelerates gradient vectors and dampens oscillations
        weight_decay=0.0005,    # penalizing large weights to prevent overfitting
        warmup_epochs=3.0,      # warm up for stablize early training
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,

        # L(total) = box * L(box) + cls * L(cls) + dfl * L(dfl)
        box=7.5,
        cls=0.5,
        dfl=1.5,    # Weight of the Distribution focal loss(modeling the probability distribution of bounding box coordinates rather than just predicting a single val)

        val=True,   # enable validation
        plots=True, # generate and save plot of training and validation metrics
        verbose=True,   # If True, displays detailed information during the validation process
        seed=0,         # random seed for training
        rect=False,
        cos_lr=False,   # help manage lr for better convergence
        close_mosaic=10,
        amp=True,       # automatic fixed precision
        fraction=1.0,   # use 100% dataset
    )

print("\n Weights saved at:", WEIGHTS_DIR)
print("last exists?", os.path.exists(LAST_CKPT))
print("best exists?", os.path.exists(BEST_CKPT))


In [ ]:
# Evaluate with diagram (learning curves + val plots)
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

EXP_DIR = os.path.join(DATASET_PATH, "runs", "yolov8x_seg_finetune")
best_model_path = os.path.join(EXP_DIR, "weights", "best.pt")
results_csv = os.path.join(EXP_DIR, "results.csv")

print("EXP_DIR:", EXP_DIR)
print("best_model:", best_model_path, "exists?", os.path.exists(best_model_path))
print("results.csv:", results_csv, "exists?", os.path.exists(results_csv))

# A. DRAW LEARNING CURVES FROM results.csv

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    def plot_cols(cols, title, ylabel):
        cols = [c for c in cols if c in df.columns]
        if not cols:
            print(f" Not found: {title}")
            return
        plt.figure()
        for c in cols:
            plt.plot(df[c].values, label=c)
        plt.title(title)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.legend()
        plt.grid(True)
        plt.show()

    # Loss curves
    loss_cols = [c for c in df.columns if ("loss" in c.lower()) and (c.startswith("train/") or c.startswith("val/"))]
    plot_cols(loss_cols, "Loss curves (train/val)", "Loss")

    # Metrics for MASK (segmentation)
    mask_metric_cols = [c for c in df.columns if c.startswith("metrics/") and "(M)" in c]
    plot_cols(mask_metric_cols, "Mask metrics curves (Precision/Recall/mAP)", "Score")

    # Metrics for BOX
    box_metric_cols = [c for c in df.columns if c.startswith("metrics/") and "(B)" in c]
    if box_metric_cols:
        plot_cols(box_metric_cols, "Box metrics curves (Precision/Recall/mAP)", "Score")
else:
    print(" Not found results.csv to draw learning curves.")

# B. RUN VAL & SHOW PLOTS (PR/F1/Confusion…)

if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    EVAL_DIR = os.path.join(EXP_DIR, "eval_plots")
    metrics = model.val(
        data=yaml_path,
        imgsz=640,
        device=0,
        project=EVAL_DIR,       # save everything here
        name="val_best",
        exist_ok=True,
        plots=True
    )

    if hasattr(metrics, "seg") and hasattr(metrics.seg, "map50"):
        print("\n Mask (Seg) metrics:")
        print(f"  seg mAP50:     {metrics.seg.map50:.4f}")
        print(f"  seg mAP50-95:  {metrics.seg.map:.4f}")
        print(f"  seg Precision: {metrics.seg.mp:.4f}")
        print(f"  seg Recall:    {metrics.seg.mr:.4f}")

    # Show plot
    out_dir = os.path.join(EVAL_DIR, "val_best")
    print("\n Val plots saved at:", out_dir)

    plot_imgs = []
    for ext in ("*.png", "*.jpg"):
        plot_imgs += glob.glob(os.path.join(out_dir, ext))
    plot_imgs = sorted(plot_imgs)

    preferred = ["confusion_matrix", "PR_curve", "F1_curve", "P_curve", "R_curve", "results"]
    preferred_imgs = []
    for key in preferred:
        preferred_imgs += [p for p in plot_imgs if key in os.path.basename(p)]

    show_list = preferred_imgs if preferred_imgs else plot_imgs
    if not show_list:
        print("Not found file plot image in folder val.")
    else:
        for p in show_list[:10]:  # limit 10 img
            print("Showing:", os.path.basename(p))
            img = Image.open(p)
            plt.figure()
            plt.imshow(img)
            plt.axis("off")
            plt.title(os.path.basename(p))
            plt.show()
else:
    print(" Not found best.pt")
